In [2]:
import pandas as pd
from collections import defaultdict
import re
import numpy as np
import ast
from functools import lru_cache
from tqdm import tqdm
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

## 1.1) Подсчёт вхождений токенов

In [2]:
RRS = pd.read_csv('Req_Res_Con_Tables/RRS.csv', index_col = False)
for col in ['Requirements', 'Responsibilities', 'Conditions']:
    RRS[col] = RRS[col].apply(ast.literal_eval)

In [3]:
count_req, count_res, count_con = defaultdict(int), defaultdict(int), defaultdict(int)
n_advertisements = RRS.shape[0]

for i in range(n_advertisements):
    for requirement in RRS.loc[i, 'Requirements']:
        count_req[requirement] += 1
    for responsibility in RRS.loc[i, 'Responsibilities']:
        count_res[responsibility] += 1
    for condition in RRS.loc[i, 'Conditions']:
        count_con[condition] += 1

In [8]:
sorted_count_req = sorted(count_req.items(), key=lambda item: item[1])
sorted_count_res = sorted(count_res.items(), key=lambda item: item[1])
sorted_count_con = sorted(count_con.items(), key=lambda item: item[1])

In [9]:
pd.DataFrame(sorted_count_req).to_csv('Requirements.csv', index = False)
pd.DataFrame(sorted_count_res).to_csv('Responsibilities.csv', index = False)
pd.DataFrame(sorted_count_con).to_csv('Conditions.csv', index = False)

## 1.2) Подсчёт вхождений пар токенов (TBD)

## 2) Обработка синонимов

In [12]:
REQUIREMENTS_TO_REPLACE = {
    '1c': ['1с.', "1c.", "1с:предприятие", "1с:erp", "1с:зуп", "1с:бухгалтерия", "1с:специалист", "1с"],
    'sql': ['sql-запрос', 'sql.', 'postgresql', 'mysql', 'pl/sql', 'mssql', 'sql.', 'nosql', 'postgresql.', 'postgres'],
    'python': ['python.', 'django'],
    'git': ['gitlab', 'git.'],
    'ci/cd': ['ci', 'ci/cd.'],
    'tcp/ip': ['tcp', 'tcpdump'],
    'kubernetes': ['kubernetes.', 'k8s'],
    'javascript': ['next.js', 'js', 'node.js', 'vue.js', 'vue'],
    'go': ['golang'],
    'c++': ['c/c++'],
    'bitrix': ['битрикс', 'битрикс24', 'bitrix24'],
    'linux': ['linux.', 'windows/linux', 'astra', 'ubuntu', 'debian', 'centos'],
    'excel': ['excel.'],
    'confluence': ['confluence.'],
    'erp': ['erp-система', 'erp.'],
    'docker': ['docker.', 'docker-compose'],
    'html': ['html5'],
    'spark': ['pyspark'],
    'http': ['https', 'http/https'],
    'agile': ['agile/scrum', 'scrum'],
    'бд': ['субд'],
    'css': ['css3']
}

RESPONSIBILITIES_TO_REPLACE = {
    'crm': ['crm-система', 'crm.'],
    'a/b-тестирование': ['a/b', 'a/b-тест'],
    'презентация': ['презентационный', 'презентовать', 'презентаций.'],
    'оптимизация': ['оптимизировать', 'оптимизации.'],
    'тестирование': ['тестировать', 'тестирования.', 'тестировщик', "тест", "автотест", "тестовый"],
    'интервью': ['интервьюирование'],
    'прогнозирование': ['прогнозировать'],
    'отчёт': ['отчётность', 'отчетности.', 'отчетность.'],
    'разработка': ['разработчик', 'разработать', 'разрабатывать', 'разработки.'],
    'внедрение': ['внедрять', 'внедрить'],
    'автоматизация': ['автоматизированный', 'автоматизировать', 'автоматизации.'],
    'анализ': ['анализировать', 'бизнес-анализ', 'исследование', 'статистика', 'исследовательский',
               'диагностика', 'диагностировать', "экспертиза", "аудит", "проверка", "валидация",
               'проверять', 'проверить'],
    'обеспечение': ['обеспечивать', 'обеспечить'],
    'планирование': ['планировать'],
    'оценка': ['оценить', 'оценивать'],
    'поддержка': ['поддерживать', 'техподдержка', 'поддержание', 'поддержки.',
                  'support'],
    'настройка': ['настраивать'],
    'проведение': ['проводить'],
    'выполнение': ['выполнить', 'выполнять'],
    'реализация': ['реализовывать', 'реализовать'],
    'проектирование': ['проектировать', 'спроектировать'],
    'обслуживание': ['обслуживать'], 
    'документация': ['документооборот', 'документировать', 'документирование', 
                 'документов.', 'документ', 'журнал', 'реестр', 'описание',
                 'описать', 'формализация', 'документацией.', 'документации.'],
    'установка': ['установить', 'установление'],
    'корректировка': ['корректировать'],
    'актуализация': ['актуализировать'],
    'визуализация': ['визуализировать'],
    'генерация': ['генерировать'],
    'инициирование': ['инициировать'],
    'масштабирование': ['масштабировать'],
    'инфраструктура': ['инфраструктурный'],
    'структурирование': ['структурировать'],
    'программа': ['программный', 'программно-аппаратный'],
    'программирование': ['программист', 'программировать'],
    'устранение': ['устранять'],
    'запуск': ['запускать'],
    'производство': ['производить'],
    'выявление': ['выявить'],
    'обработка': ['обрабатывать'],
    'расширение': ['расширить', 'расширять'],
    'распределение': ['распределять'],
    'безопасность': ['безопасный', 'кибербезопасность', 'иб.', 'иб', 'защищать',
                     'предотвращение', 'антивирусный', 'угроза'],
    'backend': ['бэкенд'],
    'frontend': ['фронтенд'],
    'обучение': ['менторство', 'онбординг', 'наставничество', 'инструктаж', 'обучить'],
    'контроль': ['мониторинг', 'наблюдение', 'отслеживание', 'трекинг', 'алертинг', 'контролировать',
                 'отслеживать'],
    'телекоммуникация': ['телекоммуникационный'],
    'коммуникация': ['коммуницировать', 'общение', 'взаимодействие', 'переговоры', 'согласование',
                     'сотрудничать', 'договариваться', 'диалог', 'согласовывать',
                     'общаться', 'обсуждать', 'переписка', 'взаимодействовать'],
    'управление': ['управлять', 'руководство', 'координировать', 'координация', 'курирование',
                   'организация', 'возглавить', 'управленческий', 'управления.', "приоритизация",
                   "декомпозировать", "декомпозиция", "постановка", 'организовывать', 'организационный', 'организовать',
                   'руководитель', 'руководящий', 'руководить'],
    'перевод': ['переводить'],
    'code-review': ['код-ревить', 'ревить', 'review'],
    'оборудование': ['оборудовать', 'оборудования.'],
    'бизнес': ['бизнес-процессов.', 'бизнеса.', 'бизнес-требование', 'бизнес-процесс', 
               'бизнес-заказчик'],
    'ai': ['искусственный', 'интеллект', 'intelligence', "нейросеть"],
    'дизайн': ['дизайн-проект', 'дизайнер', 'дизайн-система'],
    'администрирование': ['администрировать', 'администратор'],
    'продажа': ['продаж.', 'продавать'],
    'собеседование': ['собеседования.'],
    'приложение': ['приложения.', 'веб-приложение'],
    'сервис': ['web-сервис', 'сервисов.', 'веб-сервис', 'сервисный'],
    'микросервис': ['микросервисный'],
    'продукт': ['продуктовый', 'продукта.', 'продукции.', 'продукция'],
    'эксплуатация': ['эксплуатации.', 'эксплуатацию.'],
    'данные': ['данных.', 'data'],
    'процессы': ['процессов.'],
    'клиент': ['клиентов.', 'клиентский'],
    'архитектура': ['архитектурный'],
    'оформление': ['оформить', 'оформлять'],
    'подбор': ['подбирать'],
    'развозка': ['перевозка'],
    'расчёт': ['рассчитывать'],
    'монтаж': ['монтажный'],
    'написание': ['написать'],
    'бюджет': ['бюджетирование'],
    'торговый': ['торговля'],
    'аналитика': ['аналитик', 'аналитический'],
    'развёртывание': ['разворачивать']
}

CONDITIONS_TO_REPLACE = {
    'стоматология': ['стоматологией.', 'стоматологию.', 'стоматологический'],
    'дмс': ['дмс.'],
    'аккредитация': ['аккредитовать'],
    'отдых': ['отдыха.', 'отдыхать'],
    'комфортный': ['комфортабельный', 'комфортно'],
    'выплата': ['выплачивать', 'выплачиваться', 'зарплата', 'платы.'],
    'оплата': ['оплачивать'],
    'психолог': ['психологический'],
    'оздоровительный': ['оздоровление'],
    'льготный': ['льгота'],
    'кофе': ['чай/кофе', 'кофемашин', 'кофейня'],
    'самозанятость': ['самозанятой'],
    'отпуск': ['отпускной', 'отпуску.'],
    'юрист': ['юридический'],
    'лаунж-зона': ['лаунж-зоны.'],
    'питание': ['питание.', 'пища'],
    'медицина': ['мед.', 'телемедицина', 'медицинский', 'medicine'],
    'обучение': ['онлайн-обучение', 'обучения.', 'курс', 'тренинг', 'лекция'],
    '5/2': ['5-дневный', 'пятидневка', 'понедельник-пятница', 'пн-пт', '5/2.'],
    'стажировка': ['стажёр'],
    'снеки': ['печенька', 'фрукт'],
    'массаж': ['массажный'],
    'удалёнка': ['удалённый'],
    'релокация': ['релокационный', 'переезд'],
    'премия': ['поощрение'],
    'врач': ['терапевт']
}

In [13]:
def build_reverse_dict(replace_dict):
    reverse = {}

    for canonical, synonyms in replace_dict.items():
        reverse[canonical] = canonical  # сам ключ тоже заменяем на себя

        for synonym in synonyms:
            reverse[synonym] = canonical

    return reverse


REQ_REVERSE = build_reverse_dict(REQUIREMENTS_TO_REPLACE)
RESP_REVERSE = build_reverse_dict(RESPONSIBILITIES_TO_REPLACE)
COND_REVERSE = build_reverse_dict(CONDITIONS_TO_REPLACE)

In [14]:
def replace_tokens(tokens, reverse_dict):
    return list(set(
        reverse_dict.get(token, token)
        for token in tokens
    ))

Обновление матрицы RRS

In [3]:
requirements = pd.read_csv('Req_Res_Con_Tables/Requirements_filtered.csv', index_col = False)['Требование']
responsibilities = pd.read_csv('Req_Res_Con_Tables/Responsibilities_filtered.csv', index_col = False)['Обязанность']
conditions = pd.read_csv('Req_Res_Con_Tables/Conditions_filtered.csv', index_col = False)['Условие']

RRS = pd.read_csv('Req_Res_Con_Tables/RRS.csv', index_col=False)
for col in ['Requirements', 'Responsibilities', 'Conditions']:
    RRS[col] = RRS[col].apply(ast.literal_eval)

In [8]:
', '.join(list(conditions))

'full-time, медицина, студент, самозанятость, 3/3, контракт, сдельный, дорога, kion, общежитие, сберуниверситет, лаунж-зона, театр, возмещение, еда, онлайн-кинотеатр, единовременный, подработка, поход, осмотр, аккредитация, переподготовка, госпитализация, онлайн-библиотека, охрана, трансфер, пиццерия, массаж, хакатон, cкидка, дотация, вечеринка, бар, спецодежда, заболевание, семейный, переработка, 8-часовой, менторство, софинансирование, фитнес-центр, вычет, минцифра, санаторий, 1/3, йога, завтрак, операция, релокационный, покрытие, инвестиция, сберпрайм+, надбавка, бег, мерч, одежда, реферальный, продвижение, skyeng, кредитование, 24/7, кафетерий, гпх, проживание, баскетбол, юрист, отсрочка, оздоровительный, релокация, ненормированный, кредит, пн-чт, коворкинг, теннис, парк, врач, стажировка, чай, фитнес-клуб, санаторно-курортный, сменный, банковский, родственник, ресторан, лечение, спортзал, клиника, волейбол, путёвка, тимбилдинг, страховка, футбол, 2/2, парковка, кухня, удалёнка, ко

In [16]:
len(RRS)

9718

In [17]:
RRS.head()

,Id,Requirements,Responsibilities,Conditions
0,129039824,"[вопрос, такой, по, занятость, подработка, про...","[такой, по, многое, другой, подарок, подработк...","[такой, по, другой, многое, подарок, подработк..."
1,129043584,"[указывать, древесина, по, generator, refl, la...","[вес, текстура, и, создание, качественный, к, ...","[указывать, по, и, карьерный, резюме, письмо, ..."
2,128944004,"[., технический, python, нужный, и, команду., ...","[., аудит, технический, python, срока., комфор...","[по, primezone., создание, гибкий, мероприятие..."
3,128021606,"[внедрять, product, стать, по, другой, язык, з...","[культура, внедрять, product, rag, стать, по, ...","[культура, предлагать, комфортный, гибридный, ..."
4,128984804,"[nlp, фт/brd, по, задание, фреймворк, использо...","[показатель, nlp, фт/brd, по, ии-решениями., c...","[показатель, культура, по, многое, другой, пол..."


In [18]:
RRS['Requirements'] = RRS['Requirements'].apply(
    lambda x: replace_tokens(x, REQ_REVERSE)
)

RRS['Responsibilities'] = RRS['Responsibilities'].apply(
    lambda x: replace_tokens(x, RESP_REVERSE)
)

RRS['Conditions'] = RRS['Conditions'].apply(
    lambda x: replace_tokens(x, COND_REVERSE)
)

In [19]:
ids = []

req_set = set(requirements)
res_set = set(responsibilities)
con_set = set(conditions)

for _, row in RRS.iterrows():
    
    req_ok = len(set(row['Requirements']) & req_set) > 0
    res_ok = len(set(row['Responsibilities']) & res_set) > 0
    con_ok = len(set(row['Conditions']) & con_set) > 0

    if req_ok and res_ok and con_ok:
        ids.append(row['Id'])

RRS_filtered = RRS[RRS.Id.isin(ids)]
RRS_filtered.to_csv('RRS_filtered.csv', index=False)

In [20]:
len(RRS_filtered)

6759

In [22]:
RRS_filtered = pd.read_csv('Req_Res_Con_Tables/RRS_filtered.csv', index_col=False)

In [25]:
good_ids = list(RRS_filtered.Id)

Обновление текстов вакансий

In [53]:
filtered_vacancies = pd.read_csv('Data\\filtered_vacancies.csv', index_col = False)

In [54]:
correct_descriptions = {}

for vacancy in filtered_vacancies.itertuples():
    id = vacancy.Id
    if id in good_ids:
        correct_descriptions[id] = vacancy.Description

In [56]:
punctuation = set()
for text in filtered_vacancies.Description:
    new_symbols = set(re.findall(r'[^a-zA-Zа-яА-Я0-9]', text))
    punctuation |= new_symbols

NECESSARY_PUNCTUATION = ['/', '.', '-', ':', '%', '+', '#', '$']
RUBBISH_PUNCTUATION = punctuation - set(NECESSARY_PUNCTUATION)

PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES = ['/', '-', ':']

MINUSES = ['–', '─', '―', '—', '‐', '‑', '‒', '⎯']

RUBBISH_TAGS = ['<b>', '</b>', '<strong>', '</strong>', '<i>', '</i>', '<em>', '</em>', '<u>', '</u>',
                '<mark>', '</mark>', '<span>', '</span>', '<hr />', '<link />', '<div title="Change Color">',
                '<div title="Copy">', '<div title="Delete">', '<h2>', '<br />', '<ul>', '<ol>', '</h2>', '<div>', 
                '<li>', '</p>', '<p>', '</li>', 
                '</ol>', '<h4>', '<h3>', '</h3>', '</div>', '</ul>', '</h4>']

In [57]:
def rough_cleaning(text):

    cleared_text = text.replace('ё', 'е')
    cleared_text = cleared_text.replace('➕', '+')
    cleared_text = cleared_text.replace('․', '.')
    for minus in MINUSES:
        cleared_text = cleared_text.replace(minus, '-')

    for tag in RUBBISH_TAGS:
        cleared_text = cleared_text.replace(tag, ' ')
    
    for char in RUBBISH_PUNCTUATION:
        escaped_char = re.escape(char)
        cleared_text = re.sub(escaped_char, ' ', cleared_text)

    for char in PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES:
        escaped_char = re.escape(char)
        cleared_text = re.sub(rf'(\w+){escaped_char}(?=\s|$)', r'\1 ', cleared_text)
        cleared_text = re.sub(rf'(\s|^){escaped_char}(\w+)', r' \2', cleared_text)
        cleared_text = re.sub(rf'\s{escaped_char}\s', r' ', cleared_text)

    cleared_text = re.sub(r'\s+', ' ', cleared_text)
    cleared_text = cleared_text.lower().strip()

    return cleared_text

@lru_cache(maxsize=200000)
def normalize_word(word):
    return morph.parse(word)[0].normal_form

def normalize_sentence(sentence):
    words = rough_cleaning(sentence).strip().split()
    return ' '.join(normalize_word(word) for word in words)

In [58]:
for id, description in tqdm(correct_descriptions.items()):
    correct_descriptions[id] = normalize_sentence(description)

100%|██████████| 6759/6759 [00:27<00:00, 249.37it/s]


In [59]:
import pickle

with open("correct_descriptions.pkl", "wb") as f:
    pickle.dump(correct_descriptions, f)

## 2) Расчёт двухвходовых матриц релевантности

In [12]:
from east.asts import base
from tqdm import tqdm
import pandas as pd
import pickle
import numpy as np

In [4]:
RRS_filtered = pd.read_csv('Req_Res_Con_Tables/RRS_filtered.csv', index_col=False)
with open("correct_descriptions.pkl", "rb") as f:
    correct_descriptions = pickle.load(f)

In [5]:
requirements = list(pd.read_csv('Req_Res_Con_Tables/Requirements_filtered.csv', index_col = False)['Требование'])
responsibilities = list(pd.read_csv('Req_Res_Con_Tables/Responsibilities_filtered.csv', index_col = False)['Обязанность'])
conditions = list(pd.read_csv('Req_Res_Con_Tables/Conditions_filtered.csv', index_col = False)['Условие'])

In [6]:
# def clear_text(text, lowerize=True):

#     pat = re.compile(r'[^A-Za-z0-9 \-\n\r.,;!?А-Яа-яё]+')
#     cleared_text = re.sub(pat, ' ', text)

#     if lowerize:
#         cleared_text = cleared_text.lower()

#     tokens = cleared_text.split()
#     return tokens

def make_substrings(tokens, k=4):
    for i in range(max(len(tokens) - k + 1, 1)):
        yield ' '.join(tokens[i:i + k])

def get_relevance_matrix(texts, strings):

    rows = []

    for text in texts:
        ast = base.AST.get_ast(
            list(make_substrings(text.split()))
        )
        row = np.array([ast.score(s) for s in strings])
        rows.append(row)
    
    matrix = np.array(rows)

    return matrix

In [9]:
correct_descriptions_list = list(correct_descriptions.values())

### Requirements

In [24]:
requirement_relevance_matrix = get_relevance_matrix(correct_descriptions_list, requirements)
np.save('requirements_vacancies_relevance_2D_matrix.npy', requirement_relevance_matrix)

### Responsibilities

In [25]:
responsibilities_relevance_matrix = get_relevance_matrix(correct_descriptions_list, responsibilities)
np.save('responsibilities_vacancies_relevance_2D_matrix.npy', responsibilities_relevance_matrix)

### Conditions

In [26]:
conditions_relevance_matrix = get_relevance_matrix(correct_descriptions_list, conditions)
np.save('conditions_vacancies_relevance_2D_matrix.npy', conditions_relevance_matrix)

## 3) Выделение релевантных требований/обязанностей/условий

In [29]:
import numpy as np
import pandas as pd
from collections import defaultdict
import pickle
import ast

In [30]:
RRS_filtered = pd.read_csv('Req_Res_Con_Tables/RRS_filtered.csv', index_col = False)
for col in ['Requirements', 'Responsibilities', 'Conditions']:
    RRS_filtered[col] = RRS_filtered[col].apply(ast.literal_eval)

requirements_relevance_matrix = np.load('requirements_vacancies_relevance_2D_matrix.npy')
responsibilities_relevance_matrix = np.load('responsibilities_vacancies_relevance_2D_matrix.npy')
conditions_relevance_matrix = np.load('conditions_vacancies_relevance_2D_matrix.npy')

In [33]:
requirements = pd.read_csv('Req_Res_Con_Tables/Requirements_filtered.csv', index_col = False)['Требование']
responsibilities = pd.read_csv('Req_Res_Con_Tables/Responsibilities_filtered.csv', index_col = False)['Обязанность']
conditions = pd.read_csv('Req_Res_Con_Tables/Conditions_filtered.csv', index_col = False)['Условие']

### Threshold = 0.3

In [34]:
relevant_requirements_0_30 = defaultdict(set)
relevant_responsibilities_0_30 = defaultdict(set)
relevant_conditions_0_30 = defaultdict(set)

threshold = 0.3

for vacancy in RRS_filtered.itertuples():
    vac_Index, vac_id, vac_req, vac_res, vac_con = vacancy

    for index, requirement in requirements.items():
        if (requirement in vac_req) and (requirements_relevance_matrix[vac_Index, index] > threshold):
            relevant_requirements_0_30[vac_id].add(requirement)

    for index, responsibility in responsibilities.items():
        if (responsibility in vac_res) and (responsibilities_relevance_matrix[vac_Index, index] > threshold):
            relevant_responsibilities_0_30[vac_id].add(responsibility)
    
    for index, condition in conditions.items():
        if (condition in vac_con) and (conditions_relevance_matrix[vac_Index, index] > threshold):
            relevant_conditions_0_30[vac_id].add(condition)

with open("relevant_requirements_0_30.pkl", "wb") as f:
    pickle.dump(relevant_requirements_0_30, f)

with open("relevant_responsibilities_0_30.pkl", "wb") as f:
    pickle.dump(relevant_responsibilities_0_30, f)

with open("relevant_conditions_0_30.pkl", "wb") as f:
    pickle.dump(relevant_conditions_0_30, f)

### Threshold = 0.25

In [ ]:
relevant_requirements_0_25 = defaultdict(set)
relevant_responsibilities_0_25 = defaultdict(set)
relevant_conditions_0_25 = defaultdict(set)

threshold = 0.25

for vacancy in RRS_filtered.itertuples():
    vac_Index, vac_id, vac_req, vac_res, vac_con = vacancy

    for index, requirement in requirements.items():
        if (requirement in vac_req) and (requirements_relevance_matrix[vac_Index, index] > threshold):
            relevant_requirements_0_25[vac_id].add(requirement)

    for index, responsibility in responsibilities.items():
        if (responsibility in vac_res) and (responsibilities_relevance_matrix[vac_Index, index] > threshold):
            relevant_responsibilities_0_25[vac_id].add(responsibility)
    
    for index, condition in conditions.items():
        if (condition in vac_con) and (conditions_relevance_matrix[vac_Index, index] > threshold):
            relevant_conditions_0_25[vac_id].add(condition)

with open("relevant_requirements_0_25.pkl", "wb") as f:
    pickle.dump(relevant_requirements_0_25, f)

with open("relevant_responsibilities_0_25.pkl", "wb") as f:
    pickle.dump(relevant_responsibilities_0_25, f)

with open("relevant_conditions_0_25.pkl", "wb") as f:
    pickle.dump(relevant_conditions_0_25, f)